# Interpretation - Feature Importance + Partial Dependence + Project Improved Model Delivery

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>MGMT47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/blob/main/notebooks/15_interpretation_error_analysis_project_student.ipynb)

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Generate model interpretation artifacts (permutation importance, PDP/ICE)
2. Conduct error analysis to find systematic failure segments
3. Communicate model behavior honestly (limits, caveats, instability)
4. Deliver a project improved model with interpretation and error analysis
5. Use Gemini to draft explanation text, then tighten it to evidence

---

> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises**. You are expected to complete all exercises before submitting your notebook.

---

## 💼 Why This Matters: Explain Yourself

**HomeValue Analytics** calls back. They deployed your pricing model months ago, and it's working — but clients keep asking: *"Why did you value my house at \$350,000 when my neighbor's sold for \$400,000?"* Regulators are also interested: *"Can you prove your model doesn't discriminate by neighborhood demographics?"*

You need techniques that go beyond global feature importance. Partial Dependence Plots show how a single feature affects predictions across the dataset. Error analysis reveals which neighborhoods the model consistently over- or under-values — critical for knowing where to trust (and not trust) the predictions.

> **Today's focus:** Applying permutation importance, Partial Dependence Plots, and segment-level error analysis to the California Housing model, so every prediction comes with an explanation.

---

## 1. Setup

HomeValue Analytics has a champion pricing model — but clients and regulators keep asking *why* the model values one neighborhood differently from another. Answering that question requires an interpretation toolkit: `permutation_importance` measures how much each of the 8 housing features (`MedInc`, `HouseAge`, `AveRooms`, etc.) actually drives predictions, while `PartialDependenceDisplay` reveals *how* each feature affects the price estimate across its range.

The imports below load these inspection tools alongside standard regression metrics (`mean_absolute_error`, `mean_squared_error`, `r2_score`). We set `RANDOM_SEED = 474` so every importance score and PDP curve is fully reproducible, and widen the figure size to (12, 6) to accommodate feature names on horizontal bar charts.

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)

print("✓ Setup complete!")

**Reading the output:**

The `Setup complete!` confirmation with **RANDOM_SEED = 474** means all interpretation libraries loaded successfully. The two key imports — `permutation_importance` and `PartialDependenceDisplay` from scikit-learn's inspection module — are model-agnostic: they will work with the Random Forest we train next, but also with any future model HomeValue might swap in (Gradient Boosting, Ridge, etc.).

Note the figure size `(12, 6)`: wider than the default because permutation importance bar charts and PDP grids need horizontal room for feature labels. If scikit-learn is older than 1.0, the `PartialDependenceDisplay` API will differ — Colab's current environment should be fine, but this is worth checking if you reproduce the analysis locally.

**Key takeaway:** These inspection tools answer the two questions HomeValue's clients and regulators ask most: "What features matter?" (permutation importance) and "How does each feature affect the price?" (PDP/ICE). Both are computed post-training, so they do not change the model itself.

---

## 2. Load Data and Train Champion Model

HomeValue's pricing tool operates on the California Housing dataset: 20,640 census tracts described by 8 features — `MedInc` (median income), `HouseAge`, `AveRooms`, `AveBedrms`, `Population`, `AveOccup`, `Latitude`, and `Longitude`. The target `MedHouseVal` is median house value in units of $100,000, capped at 5.0 ($500,000) in the original dataset.

We split 60/20/20 and train a `RandomForestRegressor` as the champion model. Random Forests are a natural choice for this interpretation exercise because they capture non-linear relationships (so PDP curves show interesting shapes rather than flat lines) yet produce stable permutation importance estimates. The validation set is where all interpretation artifacts will be computed — never the training set, because importance measured on training data confuses memorisation with genuine predictive signal.

> 💡 **Gemini Prompt:** "Load California Housing dataset. Split into 60/20/20 train/val/test with seed 474. Print sizes."
>
> **After running, verify:**
> - Three sets: Train ~12,384, Val ~4,128, Test ~4,128
> - Target is median house value
> - Two sequential splits used
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Load dataset
california = fetch_california_housing(as_frame=True)
df = california.frame

X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']

# Split data
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.20, random_state=RANDOM_SEED)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=RANDOM_SEED)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

**Reading the output:**

The dataset splits into **Train** (~12,384), **Val** (~4,128), and **Test** (~4,128), following the standard 60/20/20 ratio. All 8 features are present: `MedInc`, `HouseAge`, `AveRooms`, `AveBedrms`, `Population`, `AveOccup`, `Latitude`, and `Longitude`.

One detail worth noting now: `MedHouseVal` is capped at 5.0 ($500,000) in the original dataset. Every property worth more than half a million dollars is recorded as 5.0. This ceiling will surface later in the error analysis as a cluster of large residuals at the high end — the model literally cannot predict values above 5.0 because no such targets exist in the training data.

**Why this matters:** Computing all interpretation artifacts on the validation set (not training data) ensures we measure genuine predictive signal. If a feature's importance drops to near zero on validation data despite being useful on training data, the model was memorising, not learning. The test set stays locked for final evaluation in a later notebook.

---

> 💡 **Gemini Prompt:** "Train a RandomForestRegressor (100 trees, max_depth=10, seed 474) on training data. Compute MAE, RMSE, R-squared on validation. Print all three metrics."
>
> **After running, verify:**
> - Model fitted on X_train and y_train
> - Three metrics printed: MAE, RMSE, R-squared
> - Predictions stored for later analysis
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Train champion model (Random Forest for interpretation)
rf_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=RANDOM_SEED, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Predictions
y_val_pred = rf_model.predict(X_val)

# Evaluation
mae = mean_absolute_error(y_val, y_val_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
r2 = r2_score(y_val, y_val_pred)

print("\n=== CHAMPION MODEL PERFORMANCE ===")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²: {r2:.4f}")

**Reading the output:**

Three metrics summarise the champion Random Forest on validation data. Typical values with 100 trees and `max_depth=10`: **MAE ≈ 0.33** ($33,000 average pricing error), **RMSE ≈ 0.48**, and **R² ≈ 0.80**. For HomeValue's CEO, these translate to: "On average, our price estimate is off by about $33,000, and the model explains 80% of the variation in house values across California."

An MAE of $33,000 on a median house worth $200,000 is a ~16% error — good enough to demonstrate that the 8 features carry genuine signal, but not yet accurate enough for high-stakes appraisals. The gap between MAE (0.33) and RMSE (0.48) tells us the error distribution has a heavy right tail: most predictions are closer than $33,000, but some are much worse. The error analysis later will reveal exactly which neighborhoods drive those outliers.

**Key takeaway:** These baseline metrics are the reference point for all subsequent interpretation. When we find a high-error segment in Section 5, we will compare its segment-specific MAE against this overall 0.33 to quantify how much worse the model performs in that segment.

---

## 3. Permutation Feature Importance

### 3.1 Compute Permutation Importance

When HomeValue's pricing team asks "What drives our model's price estimates?", permutation importance provides the answer. The method works by shuffling one feature at a time and measuring how much the model's MAE degrades. A feature whose shuffling causes a large MAE increase is one the model relies on heavily; a feature whose shuffling barely changes MAE contributes little to predictions.

> 💡 **Gemini Prompt:** "Compute permutation importance on validation set (10 repeats, neg_mean_absolute_error). Create sorted DataFrame with feature names, importance means, stds. Print full table."
>
> **After running, verify:**
> - All 8 features ranked by importance
> - Each row shows name, mean, std
> - MedInc should be top feature
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Compute permutation importance on validation set
perm_importance = permutation_importance(
    rf_model, X_val, y_val, 
    n_repeats=10, 
    random_state=RANDOM_SEED,
    scoring='neg_mean_absolute_error'
)

# Create importance DataFrame
importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance_mean': perm_importance.importances_mean,
    'importance_std': perm_importance.importances_std
}).sort_values('importance_mean', ascending=False)

print("\n=== PERMUTATION FEATURE IMPORTANCE ===")
print(importance_df)

**Reading the output:**

The permutation importance table ranks all 8 features by their impact on MAE (scored as `neg_mean_absolute_error`). **MedInc** (median income) typically dominates with importance around **0.30–0.50**, meaning shuffling income values increases HomeValue's pricing error by $30,000–$50,000. This makes economic sense: household income is the strongest determinant of housing affordability and therefore market prices.

**Latitude** and **Longitude** often rank second and third, capturing California's geographic price gradient — the coastal premium that makes a San Francisco tract worth several times more than an equivalent inland tract. Together, income and geography account for the vast majority of the model's predictive power.

Features with importance near zero — typically `Population` and `AveBedrms` — contribute almost nothing to predictions. Removing them would barely change HomeValue's MAE, which is useful information for model simplification: fewer features mean less data to collect, fewer pipeline steps, and easier explanations to clients.

The `importance_std` column shows variability across 10 shuffle repeats. A feature with high std relative to its mean has an unstable importance estimate — quote it with a confidence caveat when presenting to the pricing team.

**Key takeaway:** For HomeValue, income dominance validates the model's business logic, but heavy reliance on `Latitude`/`Longitude` raises a question the CEO should know about: are we effectively pricing based on neighborhood demographics rather than property characteristics?

---

### 3.2 Visualize Importance

A horizontal bar chart is the standard way to present permutation importance to HomeValue's stakeholders because it displays feature names legibly and lets the CEO compare magnitudes at a glance. Error bars (one standard deviation from the 10 shuffle repeats) show how stable each estimate is — a feature whose error bar overlaps zero is not significantly important, while a feature with a large bar and tight error bars is a reliably dominant driver.

> 💡 **Gemini Prompt:** "Create horizontal bar chart of permutation feature importance with error bars. Invert y-axis so most important is on top."
>
> **After running, verify:**
> - 8 features as horizontal bars with error bars
> - Most important at top
> - X-axis indicates decrease in MAE
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Plot permutation importance
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(importance_df['feature'], importance_df['importance_mean'], xerr=importance_df['importance_std'])
ax.set_xlabel('Permutation Importance (decrease in MAE)')
ax.set_title('Feature Importance - Random Forest Model')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

**Reading the output:**

The horizontal bar chart provides a visual ranking of all 8 features. **MedInc** stands out as the tallest bar by a wide margin, followed by the geographic features (`Latitude`, `Longitude`). Features like `Population` barely extend beyond zero — they are essentially noise from the model's perspective and could be dropped without meaningful accuracy loss.

Error bars that overlap zero indicate the feature's importance is not statistically distinguishable from zero across the 10 permutation repeats. For the California Housing dataset, typically 5–6 features show clearly positive importance while 2–3 cluster near zero.

**Why this matters:** This chart is the primary deliverable for HomeValue's pricing team when they ask "what drives the model?" A domain expert can verify whether the ranking makes sense (income and location driving prices is intuitive) or whether something suspicious is happening (if `Population` were the top feature, that would warrant investigation). This is also the chart regulators will review when assessing whether the model relies on potentially discriminatory proxies.

---

## 📝 PAUSE-AND-DO Exercise 1 (5 minutes)

**Task:** Create permutation importance and write 3 evidence-based bullets about feature importance.

**Instructions:**
1. Review the permutation importance results above
2. Identify the top 3 most important features
3. Write 3 evidence-based interpretation bullets

---

### YOUR INTERPRETATION HERE:

**Finding 1:**  
[Evidence-based interpretation of most important feature]

**Finding 2:**  
[Evidence-based interpretation of second feature]

**Finding 3:**  
[Evidence-based interpretation or pattern]

---

## 4. Partial Dependence Plots (PDP)

### 4.1 Create PDP for Top Features

Permutation importance told HomeValue *which* features matter — but not *how* they affect the price estimate. Partial Dependence Plots fill that gap. A PDP shows the average predicted house value as one feature varies across its range while all other features are held at their observed values. For HomeValue's pricing team, this means: "When `MedInc` rises from $20k to $80k, how does the model's price estimate respond — linearly, or with diminishing returns?"

> 💡 **Gemini Prompt:** "Select top 4 features by permutation importance. Create 2x2 partial dependence plot using PartialDependenceDisplay.from_estimator with kind='average'."
>
> **After running, verify:**
> - Four subplots, one per top feature
> - Suptitle reads 'Partial Dependence Plots'
> - Each shows marginal effect on prediction
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Select top 4 features for PDP
top_features = importance_df.head(4)['feature'].tolist()

print(f"Creating PDP for: {top_features}")

# Create partial dependence plots
fig, ax = plt.subplots(figsize=(14, 10))
display = PartialDependenceDisplay.from_estimator(
    rf_model, X_val, top_features, 
    ax=ax, 
    kind="average",
    random_state=RANDOM_SEED
)
plt.suptitle('Partial Dependence Plots - Top 4 Features', fontsize=16)
plt.tight_layout()
plt.show()

**Reading the output:**

The four subplots each show how predicted `MedHouseVal` changes as one feature varies. For **MedInc**, expect a strong positive curve: predicted prices rise steeply from income 0–5 (tracts where median household income is $0–$50k) and then flatten above income 8–10 ($80k–$100k). The flattening reflects both the $500,000 target cap and the scarcity of ultra-high-income tracts in the training data.

For **Latitude** and **Longitude**, the PDPs capture California's geographic price gradient. Lower latitude (Southern California) and specific longitude ranges (coastal strip) correlate with higher predictions. These curves are non-monotonic — a small coordinate shift can cross from an expensive coastal city to a cheaper inland valley — which is exactly why the Random Forest outperforms linear models on this data.

For **HouseAge** or **AveRooms** (depending on the importance ranking), the relationship may be U-shaped or flat, showing that these features have a weaker or more complex effect on pricing.

**Key takeaway:** PDPs transform HomeValue's black-box Random Forest into interpretable feature-response curves. If a client asks "Why was my house valued lower than my neighbor's?", the pricing team can point to these curves and say "Income levels in your tract drive the estimate more than house age does — here's the evidence."

---

### 4.2 Individual Conditional Expectation (ICE) Plots

PDP shows the *average* effect of `MedInc` across all 4,128 validation tracts. But HomeValue's pricing team wants to know: does income affect pricing the same way in coastal San Francisco as in inland Bakersfield? ICE plots answer this by drawing a separate line for *each individual tract*. The thick PDP line is the average of all ICE curves.

If the ICE curves are tightly bundled, income has a consistent effect everywhere. If they fan out or cross, the income effect depends on context (e.g., geography) — a signal of feature interactions that the single PDP line would hide.

> 💡 **Gemini Prompt:** "Create ICE plot for the most important feature using PartialDependenceDisplay with kind='both' (individual curves alpha=0.1 + average PDP line). Print interpretation notes."
>
> **After running, verify:**
> - Many semi-transparent individual curves + bold average line
> - Suptitle identifies the feature
> - Interpretation notes about feature interactions
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Create ICE plots for top feature
top_feature = top_features[0]

fig, ax = plt.subplots(figsize=(10, 6))
display = PartialDependenceDisplay.from_estimator(
    rf_model, X_val, [top_feature],
    kind="both",  # Shows both average PDP and individual ICE curves
    ax=ax,
    random_state=RANDOM_SEED,
    ice_lines_kw={"alpha": 0.1}
)
plt.suptitle(f'ICE Plot for {top_feature}', fontsize=14)
plt.tight_layout()
plt.show()

print(f"\n⚠️ Interpretation Notes:")
print(f"  - PDP shows average effect of {top_feature} on predictions")
print(f"  - ICE curves show effect for individual samples")
print(f"  - Wide spread in ICE = interactions with other features")

**Reading the output:**

The ICE plot overlays hundreds of thin, semi-transparent lines (individual tracts) with a thick solid PDP average. For **MedInc**, most ICE curves follow the same upward trend, but you can see spread: some curves are steeper (coastal tracts where income amplifies the geographic premium) and some are flatter (inland tracts where income has a smaller marginal effect on prices).

Where ICE curves *cross* each other, that is strong evidence of a **feature interaction**. For HomeValue, this means the income-price relationship is not one-size-fits-all — a $10k income increase in a beachfront zip code boosts the predicted price more than the same increase in a rural area. This heterogeneity matters for client communication: quoting a single PDP curve can be misleading if the client's tract falls in a region where the relationship is different from the average.

**Why this matters:** Before HomeValue makes policy recommendations based on PDP curves (e.g., "increasing neighborhood income by $10k raises estimated home values by $X"), ICE plots reveal whether that relationship holds uniformly or varies by subgroup. If the spread is wide, the pricing team should qualify their statements with segment-level caveats.

---

## 5. Error Analysis

### 5.1 Residual Analysis

Aggregate metrics tell HomeValue's CEO that the model is "off by $33,000 on average" — but they do not reveal *where* or *why*. Residual analysis digs into the pattern of errors. A residual plot (predicted vs. error) reveals systematic failures: a funnel shape means errors grow with predicted value (bad news for the luxury segment), clusters of large residuals point to neighborhoods the model consistently misprices, and non-zero mean residuals signal systematic over- or under-valuation.

The two-panel visualization below shows residuals vs. predicted values (left) and the residual distribution (right). HomeValue wants to see residuals randomly scattered around zero with constant variance — any deviation from that pattern is a deployment risk worth investigating.

> 💡 **Gemini Prompt:** "Compute residuals (actual - predicted) on validation. Create 1x2 subplot: scatter of residuals vs predicted (left) and histogram of residuals (right, 50 bins). Print mean, std, median."
>
> **After running, verify:**
> - Scatter centered around zero with red reference line
> - Histogram shows distribution shape
> - Residual statistics printed below
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Compute residuals
residuals = y_val - y_val_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residual plot
axes[0].scatter(y_val_pred, residuals, alpha=0.3)
axes[0].axhline(y=0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted Values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residual Plot')

# Residual histogram
axes[1].hist(residuals, bins=50, edgecolor='black')
axes[1].set_xlabel('Residuals')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Residual Distribution')

plt.tight_layout()
plt.show()

print(f"\nResidual Statistics:")
print(f"  Mean: {residuals.mean():.4f}")
print(f"  Std: {residuals.std():.4f}")
print(f"  Median: {residuals.median():.4f}")

**Reading the output:**

The **left panel** (residuals vs. predicted values) reveals a telltale **funnel shape**: residuals are tight for low predicted values ($50k–$150k) but fan out dramatically for high predictions ($300k–$500k). This heteroscedasticity means HomeValue's pricing accuracy depends heavily on the price segment — the luxury neighborhoods that generate the highest commissions are precisely where the model is least reliable.

You may also notice a cluster of large positive residuals near predicted value 5.0, caused by the $500,000 target cap. The model predicts 5.0 for a $500k tract and a $2M mansion alike, creating irreducible error for everything above the cap.

The **right panel** (residual histogram) is roughly symmetric and centred near zero (mean ≈ 0.00), confirming no systematic bias overall. But the right tail is heavier than the left — the model under-predicts expensive homes more often than it over-predicts cheap ones.

**Why this matters:** This funnel pattern is HomeValue's primary deployment risk. The pricing team needs to decide: deploy with a confidence disclaimer for high-value properties, build a separate luxury-segment model with richer features, or invest in collecting property-level data (lot size, renovations, school ratings) that the current 8-feature dataset lacks.

---

### 5.2 Segment Error Analysis

The residual plot showed HomeValue that errors are not evenly distributed. Segment error analysis makes this precise by splitting the validation set into quartiles of `MedInc` (the top feature) and computing MAE separately for each group. If Q4 (highest-income tracts) has 2–3x the error of Q1 (lowest-income tracts), that segment represents a systematic failure mode that HomeValue must either fix (better features, specialised model) or disclose (confidence intervals, usage caveats).

> 💡 **Gemini Prompt:** "Segment validation data into quartiles by most important feature. For each quartile, compute mean, median, std of absolute errors. Print segment error table."
>
> **After running, verify:**
> - Table has 4 rows (Q1-Q4) with error stats
> - Higher segments may show larger errors
> - Sample counts roughly equal per quartile
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Analyze errors by segments
# Create price segments
X_val_analysis = X_val.copy()
X_val_analysis['y_true'] = y_val.values
X_val_analysis['y_pred'] = y_val_pred
X_val_analysis['abs_error'] = np.abs(residuals)

# Segment by top feature
top_feature = importance_df.iloc[0]['feature']
X_val_analysis[f'{top_feature}_segment'] = pd.qcut(X_val_analysis[top_feature], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])

# Error by segment
segment_errors = X_val_analysis.groupby(f'{top_feature}_segment').agg({
    'abs_error': ['mean', 'median', 'std'],
    'y_true': 'count'
}).round(4)

print(f"\n=== ERROR ANALYSIS BY {top_feature} SEGMENT ===")
print(segment_errors)

**Reading the output:**

The segment error table breaks down MAE, median error, error standard deviation, and sample count for each `MedInc` quartile. The typical pattern on California Housing: **Q1** (lowest income, ~$0–$25k median) has low MAE because these affordable tracts cluster in a narrow price range that is easy to predict. **Q4** (highest income, ~$50k+) has MAE 2–3x higher, often around $45,000–$65,000, because high-income areas contain diverse property types — $400k condos and $1.5M estates sharing the same tract — that 8 features cannot distinguish.

Sample counts should be roughly equal across quartiles (~1,032 each), confirming that `pd.qcut` created balanced groups rather than one segment dominating the analysis.

**Key takeaway:** HomeValue now has a quantitative answer to "Where does the model struggle?" The Q4 segment — the luxury market where commissions are highest — is where the pricing tool is least trustworthy. Any effort to improve the model should focus here: richer features, removing the $500k cap, or training a separate specialist model for high-income tracts.

---

> 💡 **Gemini Prompt:** "Create box plot of absolute errors per quartile segment of the most important feature."
>
> **After running, verify:**
> - 4 boxes (Q1-Q4) showing error distributions
> - Outliers visible beyond whiskers
> - X-axis labels feature quartile, y-axis absolute error
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Visualize errors by segment
fig, ax = plt.subplots(figsize=(10, 6))
X_val_analysis.boxplot(column='abs_error', by=f'{top_feature}_segment', ax=ax)
ax.set_xlabel(f'{top_feature} Quartile')
ax.set_ylabel('Absolute Error')
ax.set_title(f'Error Distribution by {top_feature} Segment')
plt.suptitle('')  # Remove default title
plt.tight_layout()
plt.show()

**Reading the output:**

The box plot visualises what the table quantified. Each box shows the interquartile range of absolute errors; the line inside is the median; whiskers extend to 1.5x IQR; outlier dots beyond the whiskers are the worst individual mispricings.

The **Q4 box** is visibly taller and higher than Q1–Q3, confirming the model's luxury-segment weakness. Q4 also has more outliers — some tracts with absolute errors exceeding $100,000, which would be catastrophic in a client-facing valuation. Compare medians: Q1 and Q2 sit near $15,000–$25,000 (acceptable), while Q4's median is $40,000–$60,000 (problematic for a pricing tool clients pay for).

**Why this matters:** This box plot is a powerful communication tool for HomeValue's leadership. Instead of reporting a single MAE of ~$33,000, the pricing team can say: "We predict affordable neighborhoods within ~$20,000, but luxury neighborhoods only within ~$50,000. We recommend adding property-level features or building a luxury-specific model before using this tool for high-value appraisals."

---

## 📝 PAUSE-AND-DO Exercise 2 (5 minutes)

**Task:** Run segment error analysis and identify one failure segment.

**Instructions:**
1. Review the segment error analysis above
2. Identify which segment has the highest error
3. Propose a hypothesis for why this segment performs poorly
4. Suggest one potential improvement

---

### YOUR SEGMENT ANALYSIS HERE:

**Highest Error Segment:**  
[Which segment and why?]

**Hypothesis:**  
[Why does this segment have higher errors?]

**Potential Improvement:**  
[What could help reduce errors in this segment?]

---

## 6. Interpretation Narrative Template (Evidence-Based)

### 6.1 Model Strengths

Every interpretation report HomeValue delivers to clients or presents to the board should follow a consistent structure: strengths (backed by metrics), limitations (backed by error analysis), and recommendations (backed by both). The templates below show how to write each section using evidence from the analyses above — not opinions, but numbers.

**Model Strengths (Evidence-Based):**

1. **Overall Accuracy**: The Random Forest achieves an R² of [X.XX] on validation, explaining [XX]% of variance in California housing prices — a substantial improvement over a mean-baseline predictor.

2. **Key Drivers Are Intuitive**: Permutation importance confirms that `MedInc` (importance ≈ [X.XX]) is the dominant predictor, followed by geographic features (`Latitude`, `Longitude`). This aligns with domain knowledge: income and location drive home prices.

3. **Non-Linear Patterns Captured**: PDP curves show that the income-price relationship is not linear — prices rise steeply at low income levels and flatten at high levels. The Random Forest captures this curve; a linear model would miss it.

---

### 6.2 Model Limitations

Honest limitation reporting is what separates HomeValue from competitors who oversell their models. The board and regulators will respect candour — and clients are better served by a tool that says "I'm uncertain here" than one that silently misvalues a $2M property.

**Model Limitations (Honest Assessment):**

1. **Luxury-Segment Weakness**: Error analysis reveals that Q4 (highest-income tracts) has MAE of [X.XX] ($[XX],000), roughly [X]x the Q1 error. High-value properties are where HomeValue's pricing tool is least reliable.

2. **Feature Interactions**: ICE plots show substantial spread in individual effects for `MedInc`, suggesting the income-price relationship varies by geography. Quoting a single average relationship to all clients is misleading.

3. **Target Cap at $500,000**: The dataset caps `MedHouseVal` at 5.0, making all properties above $500k indistinguishable. This creates irreducible error in the luxury segment that no model architecture can fix without uncapped data.

4. **Census-Tract Granularity**: Features are aggregated at the census-tract level, not individual-property level. Two very different homes in the same tract receive similar predictions, limiting accuracy for property-specific valuations.

---

### 6.3 Recommendations

Recommendations bridge the gap between analysis and action. Each recommendation below is tied to a specific finding from the error analysis, so stakeholders can trace the logic from evidence to proposed action.

**Recommendations:**

1. **Deploy with Segment-Specific Disclaimers**: For Q4 tracts (high income), flag predictions as "lower confidence" and display wider error bounds to clients. The pricing team should manually review any estimate above $400k before it reaches a client.

2. **Invest in Property-Level Features**: Adding lot size, renovation year, school district ratings, and coastal distance would help distinguish properties within the same high-income tract — directly attacking the Q4 error.

3. **Monitor Permutation Importance Over Time**: If the feature importance ranking shifts (e.g., `HouseAge` suddenly becoming the top predictor), that signals a data or market change that warrants model retraining.

4. **Report Prediction Intervals, Not Point Estimates**: When communicating prices to clients, provide a range (e.g., "$310k–$370k") rather than a single number. The range should widen for Q4 properties and narrow for Q1–Q2.

---

## 7. Project Milestone 3 Scaffold

### 7.1 Deliverable Checklist

For your course project, Milestone 3 requires you to apply the same interpretation and error analysis techniques from this HomeValue example to your own dataset. The checklist below maps each required deliverable to the section of this notebook where you learned the technique.

**Project Milestone 3 Requirements:**

- [ ] Updated model comparison table (baseline vs improved models) — from NB14
- [ ] Champion model selection with justification — from NB14
- [ ] Permutation importance plot and interpretation — Section 3 of this notebook
- [ ] PDP/ICE plots for top 3–4 features — Section 4 of this notebook
- [ ] Segment error analysis with findings — Section 5 of this notebook
- [ ] Evidence-based interpretation narrative — Section 6 template
- [ ] Model limitations section (honest assessment) — Section 6.2 template
- [ ] Next steps and recommendations — Section 6.3 template

---

## 8. Wrap-Up: Key Takeaways

### What We Learned Today:

1. **Permutation Importance**: Model-agnostic method to measure feature importance
2. **Partial Dependence**: Visualizing average effect of features on predictions
3. **ICE Plots**: Understanding individual-level feature effects and interactions
4. **Error Analysis**: Finding systematic failure patterns through segmentation
5. **Honest Communication**: Evidence-based interpretation with clear limitations

### Interpretation Best Practices:

- ✓ Always compute importance on validation/test data, not training data
- ✓ Report standard deviations to show stability
- ✓ Check for correlated features when interpreting importance
- ✓ Use PDP cautiously when features are highly correlated
- ✓ Conduct segment analysis to find failure modes
- ✓ Be honest about limitations and uncertainties

### Remember:

> **"Interpretation is about honest communication, not selling the model."**  
> Report what you find, including limitations and failure modes.

---

## Participation Assignment Submission Instructions

### To Submit This Notebook:

1. **Complete all exercises**: Fill in both PAUSE-AND-DO exercise cells with your findings
2. **Run All Cells**: Execute `Runtime → Run all` to ensure everything works
3. **Save a Copy**: `File → Save a copy in Drive or Download the .ipynb extension`
4. **Submit**: Upload your `.ipynb` file in the participation assignment you find in the course Brightspace page.

### Before Submitting, Check:

- [ ] All cells execute without errors
- [ ] All outputs are visible
- [ ] Both exercise responses are complete
- [ ] Notebook is shared with correct permissions
- [ ] You can explain every line of code you wrote

### Next Step:

Complete the **Quiz** in Brightspace (auto-graded)

---

## Bibliography

- scikit-learn User Guide: [Inspection Tools](https://scikit-learn.org/stable/inspection.html) (permutation importance, partial dependence)
- Molnar, C. (2022). *Interpretable Machine Learning*. [Online book](https://christophm.github.io/interpretable-ml-book/)
- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An Introduction to Statistical Learning with Python* (ISLP). Springer.
- Breiman, L. (2001). "Random Forests." *Machine Learning*, 45(1), 5-32.

---




<center>

Thank you!

</center>